# Run the red-team harness on a Colab GPU

Runtime → Change runtime type → **A100 GPU** (L4/T4 also work for 7–8B models).
Everything runs inside this VM: Ollama serves the models on the GPU, and the harness talks to it at `localhost:11434`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
# Ollama's installer needs zstd on Colab images
apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
%%bash
# Keep both models resident (A100 has room) and serve in the background
export OLLAMA_MAX_LOADED_MODELS=2 OLLAMA_KEEP_ALIVE=1h
nohup ollama serve > /content/ollama.log 2>&1 &
sleep 5 && curl -s localhost:11434/api/version

In [ ]:
!ollama pull llama3:8b && ollama pull dolphin-mistral

In [ ]:
%%bash
cd /content
[ -d multi-agent-red-team ] || git clone https://github.com/guynutman/multi-agent-red-team.git
cd multi-agent-red-team
pip -q install uv
uv sync -q
uv run pytest -q   # offline tests; Ollama tests run too now that the server is up

## Campaign
Adjust `BUDGET` and sub-agents. With both models on the GPU, each attempt takes seconds rather than minutes.

In [ ]:
%%bash
cd /content/multi-agent-red-team
BUDGET=50
uv run python main.py \
  --campaign-name colab_full \
  --target ollama:llama3:8b \
  --attacker ollama:dolphin-mistral \
  --sub-agents basic_direct,external_indirect,system_prompt_leak,secret_leak,tool_hijack,crescendo_leak \
  --budget $BUDGET --max-turns 4 \
  --db-path campaigns.db \
  --output-report reports/colab_full.md

In [ ]:
from IPython.display import Markdown
Markdown(open('/content/multi-agent-red-team/reports/colab_full.md').read()[:20000])

In [ ]:
from google.colab import files
files.download('/content/multi-agent-red-team/reports/colab_full.md')
files.download('/content/multi-agent-red-team/campaigns.db')

### Bigger models
An A100 40GB fits a 4-bit 70B target alone (`ollama pull llama3.1:70b`, ~40 GB), so pair it with a hosted attacker/judge, e.g. `--attacker gemini:gemini-flash-latest` after setting `GEMINI_API_KEY` in Colab secrets → `os.environ`.